# TALMAS: Timestep-Adaptive, Layer-Dependent Masked Attention Suppression
**Training-free inference intervention on LLaDA-8B**

Runtime: Google Colab Pro, A100 40 GB  
Run all cells top-to-bottom.

In [ ]:
# Cell 0 — Install Dependencies
# Pin transformers <4.44: LLaDA's custom model class (LLaDAModelLM) predates the
# all_tied_weights_keys requirement introduced in transformers 4.44.
!pip install -q "transformers>=4.38.0,<4.44.0" accelerate datasets evaluate tqdm pandas matplotlib
!pip install -q human-eval  # for HumanEval pass@k metric

# ⚠️  After this cell completes, do:  Runtime → Restart session
# Then run all cells from Cell 1 downward.

In [ ]:
# Cell 1 — Imports & Config
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm.auto import tqdm
import copy, json, re, time

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2 — Load LLaDA-8B

# ── Compatibility patch (idempotent — safe to re-run) ─────────────────────────
# transformers ≥4.44 checks model.all_tied_weights_keys during from_pretrained.
# LLaDA's custom class predates that requirement. Patch __init_subclass__ on
# PreTrainedModel so any subclass loaded via trust_remote_code gets the attr.
# Idempotency guard prevents recursion if this cell is re-run.
from transformers.modeling_utils import PreTrainedModel as _PTM

if not getattr(_PTM, "_talmas_isc_patched", False):
    @classmethod
    def _auto_add_tied_keys(cls, **kwargs):
        # object.__init_subclass__ accepts no kwargs; call safely
        try:
            object.__init_subclass__(**kwargs)
        except TypeError:
            pass
        if not hasattr(cls, "all_tied_weights_keys"):
            cls.all_tied_weights_keys = []

    _PTM.__init_subclass__ = _auto_add_tied_keys
    _PTM._talmas_isc_patched = True
    print("Compatibility patch applied.")
else:
    print("Compatibility patch already applied — skipping.")
# ─────────────────────────────────────────────────────────────────────────────

MODEL_ID = "GSAI-ML/LLaDA-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model.eval()

# ── Resolve MASK token ID ─────────────────────────────────────────────────────
# LLaDA adds [MASK] to the LLaMA vocabulary but doesn't always register it as
# tokenizer.mask_token. Try several lookup strategies in order.
MASK_TOKEN_ID = tokenizer.mask_token_id

if MASK_TOKEN_ID is None:
    _unk = tokenizer.unk_token_id
    for _candidate in ("[MASK]", "<mask>", "▁[MASK]"):
        _id = tokenizer.convert_tokens_to_ids(_candidate)
        if _id is not None and _id != _unk:
            MASK_TOKEN_ID = _id
            tokenizer.mask_token = _candidate
            tokenizer.mask_token_id = _id
            print(f"Resolved mask token: '{_candidate}' → id {_id}")
            break

if MASK_TOKEN_ID is None and hasattr(model.config, "mask_token_id"):
    MASK_TOKEN_ID = model.config.mask_token_id
    print(f"Resolved mask token from model.config: id {MASK_TOKEN_ID}")

if MASK_TOKEN_ID is None:
    print("Special tokens map:", tokenizer.special_tokens_map)
    print("All special token ids:", tokenizer.all_special_ids)
    raise AssertionError(
        "Could not resolve MASK_TOKEN_ID automatically. "
        "Inspect output above and set MASK_TOKEN_ID manually below."
    )

print(f"MASK_TOKEN_ID: {MASK_TOKEN_ID}")
print(f"num_hidden_layers: {model.config.num_hidden_layers}")

In [ ]:
# Cell 3 — TALMAS Core Implementation

def f_timestep(x: float) -> float:
    return x ** 2

def g_layer(layer_idx: int, num_layers: int) -> float:
    u = layer_idx / num_layers
    return torch.sigmoid(torch.tensor(8.0 * (u - 0.5))).item()

def compute_lambda(lambda_max, r_t, layer_idx, num_layers):
    return lambda_max * f_timestep(1.0 - r_t) * g_layer(layer_idx, num_layers)

def talmas_logit_bias(attn_logits, mask_positions, lam, mu):
    if lam == 0.0:
        return attn_logits
    m = mask_positions.float()
    m_key   = m.unsqueeze(1).unsqueeze(2)
    m_query = m.unsqueeze(1).unsqueeze(3)
    return attn_logits - lam * m_key * ((1.0 - m_query) + mu * m_query)


def _find_attention_modules(model):
    """Find (name, module, layer_idx) for each block that calls SDPA."""
    candidates = []
    for name, module in model.named_modules():
        if name.endswith('.self_attn'):
            candidates.append((name, module))
    if candidates:
        return [(n, m, i) for i, (n, m) in enumerate(candidates)]
    for name, module in model.named_modules():
        parts = name.split('.')
        if parts[-1].isdigit() and len(parts) >= 2 and parts[-2] == 'blocks':
            candidates.append((name, module))
    candidates.sort(key=lambda x: int(x[0].split('.')[-1]))
    return [(n, m, i) for i, (n, m) in enumerate(candidates)]


def talmas_cleanup(model):
    """Force-remove any stale TALMAS patches from all attention blocks."""
    removed = 0
    for name, module in model.named_modules():
        if hasattr(module, '_talmas_orig_fwd'):
            module.forward = module._talmas_orig_fwd
            del module._talmas_orig_fwd
            removed += 1
    if removed:
        print(f"talmas_cleanup: restored {removed} blocks")


class TALMASHookManager:
    def __init__(self, model, lambda_max, mu,
                 use_timestep_gate=True, use_layer_gate=True):
        self.model = model
        self.lambda_max = lambda_max
        self.mu = mu
        self.use_timestep_gate = use_timestep_gate
        self.use_layer_gate = use_layer_gate
        self.r_t = 1.0
        self.mask_positions = None
        self.hooks = []
        self.num_layers = (model.config.num_hidden_layers
                           if hasattr(model.config, 'num_hidden_layers') else 32)
        self._register_hooks()
        print(f"TALMASHookManager: {len(self.hooks)}/{self.num_layers} blocks patched, "
              f"lambda_max={lambda_max}, mu={mu}")

    def _patch_attention(self, module, layer_idx):
        manager = self
        import functools
        original_sdpa = torch.nn.functional.scaled_dot_product_attention

        @functools.wraps(original_sdpa)
        def patched_sdpa(query, key, value, attn_mask=None, **kw):
            # Early exit: baseline or no state set
            if manager.mask_positions is None or manager.lambda_max == 0.0:
                return original_sdpa(query, key, value, attn_mask=attn_mask, **kw)

            S_q = query.shape[2]
            S_k = key.shape[2]
            mask_pos = manager.mask_positions.to(query.device)

            # Stale-state guard: if mask_positions length != current sequence, skip bias
            if mask_pos.shape[1] != S_q or mask_pos.shape[1] != S_k:
                return original_sdpa(query, key, value, attn_mask=attn_mask, **kw)

            if manager.use_timestep_gate and manager.use_layer_gate:
                lam = compute_lambda(manager.lambda_max, manager.r_t,
                                     layer_idx, manager.num_layers)
            elif manager.use_timestep_gate:
                lam = manager.lambda_max * f_timestep(1.0 - manager.r_t)
            elif manager.use_layer_gate:
                lam = manager.lambda_max * g_layer(layer_idx, manager.num_layers)
            else:
                lam = manager.lambda_max

            m_key    = mask_pos.float().unsqueeze(1).unsqueeze(2)   # (B,1,1,S)
            m_query  = mask_pos.float().unsqueeze(1).unsqueeze(3)   # (B,1,S,1)
            query_gate = (1.0 - m_query) + manager.mu * m_query
            bias = -(lam * m_key * query_gate)                       # (B,1,S,S)

            # Only combine if attn_mask shape is compatible
            if attn_mask is not None and attn_mask.shape[-2:] == bias.shape[-2:]:
                combined = attn_mask + bias
            else:
                combined = bias
            return original_sdpa(query, key, value, attn_mask=combined, **kw)

        # Store original fwd on module itself for global cleanup
        if not hasattr(module, '_talmas_orig_fwd'):
            module._talmas_orig_fwd = module.forward

        module._talmas_patched_sdpa = patched_sdpa
        _orig_fwd = module.forward
        self.hooks.append((module, _orig_fwd))

        def _wrapped_fwd(*args, **kwargs):
            old = torch.nn.functional.scaled_dot_product_attention
            torch.nn.functional.scaled_dot_product_attention = module._talmas_patched_sdpa
            try:
                return _orig_fwd(*args, **kwargs)
            finally:
                torch.nn.functional.scaled_dot_product_attention = old

        module.forward = _wrapped_fwd

    def set_state(self, r_t, mask_positions):
        self.r_t = r_t
        self.mask_positions = mask_positions

    def remove(self):
        for module, orig in self.hooks:
            module.forward = orig
            # Also clear the persistent backup so talmas_cleanup knows it's clean
            if hasattr(module, '_talmas_orig_fwd'):
                del module._talmas_orig_fwd
        self.hooks.clear()
        print("TALMASHookManager: hooks removed")

    def _register_hooks(self):
        for name, module, layer_idx in _find_attention_modules(self.model):
            self._patch_attention(module, layer_idx)


print("Gate check:")
print(f"  f(0.0)={f_timestep(0.0):.2f}  f(1.0)={f_timestep(1.0):.2f}")
print(f"  g(0,32)={g_layer(0,32):.3f}  g(31,32)={g_layer(31,32):.3f}")

In [ ]:
# Cell 3b — Hook Diagnostic + Bias Verification
import inspect

# ── 1. Confirm hooks registered ──────────────────────────────────────────────
_test_mgr = TALMASHookManager(model, lambda_max=4.0, mu=0.0,
                               use_timestep_gate=False, use_layer_gate=False)
print(f"Hooks registered: {len(_test_mgr.hooks)}/{model.config.num_hidden_layers}")
_test_mgr.remove()

# ── 2. Verify bias changes logits ────────────────────────────────────────────
# Run one forward pass with and without TALMAS on a masked input.
# If TALMAS is working, the logits must differ.
_test_prompt = "The answer is"
_test_ids = tokenizer(_test_prompt, return_tensors="pt").input_ids.to(model.device)
_gen_len = 8

# Build input: prompt + masked generation positions
_masked = torch.full((1, _gen_len), MASK_TOKEN_ID, dtype=torch.long, device=model.device)
_full_ids = torch.cat([_test_ids, _masked], dim=1)
_is_masked = (_full_ids == MASK_TOKEN_ID)

# Baseline logits
with torch.no_grad():
    _base_logits = model(input_ids=_full_ids).logits.cpu()

# TALMAS logits (Static Bias, lambda=4.0)
_mgr = TALMASHookManager(model, lambda_max=4.0, mu=0.0,
                          use_timestep_gate=False, use_layer_gate=False)
_mgr.set_state(r_t=0.0, mask_positions=_is_masked)  # r_t=0 → full bias applied
with torch.no_grad():
    _talmas_logits = model(input_ids=_full_ids).logits.cpu()
_mgr.remove()

_diff = (_talmas_logits - _base_logits).abs()
print(f"\nLogit diff (TALMAS vs Baseline):")
print(f"  max={_diff.max():.4f}  mean={_diff.mean():.4f}  nonzero={(_diff > 1e-6).sum().item()}")
if _diff.max() > 1e-4:
    print("  ✓ TALMAS bias IS being applied — logits differ")
else:
    print("  ✗ Logits identical — bias NOT applied (check mask_positions shape)")

In [ ]:
# Cell 4 — LLaDA Inference Loop

@torch.no_grad()
def llada_generate(
    model,
    tokenizer,
    prompt_ids: torch.Tensor,           # (1, prompt_len)
    gen_len: int = 128,
    num_steps: int = 20,
    hook_manager: TALMASHookManager = None,
) -> torch.Tensor:
    """
    LLaDA reverse diffusion generation.
    Returns generated token ids (1, gen_len).
    """
    device = prompt_ids.device
    B = prompt_ids.shape[0]

    # Initialize: all generation positions are [MASK]
    gen_ids = torch.full((B, gen_len), MASK_TOKEN_ID,
                         dtype=torch.long, device=device)
    input_ids = torch.cat([prompt_ids, gen_ids], dim=1)  # (B, prompt+gen)

    prompt_len = prompt_ids.shape[1]

    for step in range(num_steps - 1, -1, -1):
        # r_t = fraction of tokens still to be masked at this step
        r_t = step / max(num_steps - 1, 1)

        # Identify masked positions across full sequence
        is_masked = (input_ids == MASK_TOKEN_ID)  # (B, total_len)

        # Update hook state
        if hook_manager is not None:
            hook_manager.set_state(r_t=r_t, mask_positions=is_masked)

        # Forward pass through LLaDA
        logits = model(input_ids=input_ids).logits  # (B, total_len, vocab)

        # How many generation tokens to unmask this step
        num_masked = is_masked[:, prompt_len:].sum(dim=1)  # (B,)
        target_masked = int(round(r_t * gen_len))
        num_to_unmask = max(0, int(num_masked[0].item()) - target_masked)

        if num_to_unmask > 0:
            gen_logits = logits[:, prompt_len:, :]   # (B, gen_len, vocab)
            gen_probs  = F.softmax(gen_logits, dim=-1)

            # Pick top-confidence masked positions to unmask
            gen_conf = gen_probs[0].max(dim=-1).values   # (gen_len,)
            still_masked = is_masked[0, prompt_len:]
            conf_masked = gen_conf.clone()
            conf_masked[~still_masked] = -1.0
            _, top_idx = conf_masked.topk(num_to_unmask)

            new_tokens = gen_probs[0].argmax(dim=-1)  # greedy decode
            input_ids[0, prompt_len + top_idx] = new_tokens[top_idx]

    return input_ids[:, prompt_len:]


print("llada_generate defined.")

In [ ]:
# Cell 5 — Ablation Configurations

ABLATION_CONFIGS = [
    {
        "id": 1,
        "name": "Baseline (LLaDA)",
        "lambda_max": 0.0,
        "use_timestep_gate": False,
        "use_layer_gate": False,
        "mu": 0.0,
        "description": "lambda_max=0 — recovers original LLaDA exactly"
    },
    {
        "id": 2,
        "name": "Static Bias",
        "lambda_max": 4.0,   # tune via sweep
        "use_timestep_gate": False,
        "use_layer_gate": False,
        "mu": 0.0,
        "description": "Fixed lambda, no timestep or layer gating"
    },
    {
        "id": 3,
        "name": "Timestep-Only",
        "lambda_max": 4.0,
        "use_timestep_gate": True,
        "use_layer_gate": False,   # g(l/L) = 1 (uniform)
        "mu": 0.0,
        "description": "Quadratic timestep ramp, uniform across layers"
    },
    {
        "id": 4,
        "name": "Layer-Only",
        "lambda_max": 4.0,
        "use_timestep_gate": False,  # f(1-r_t) = 1 (uniform)
        "use_layer_gate": True,
        "mu": 0.0,
        "description": "Sigmoid layer gate, uniform across timesteps"
    },
    {
        "id": 5,
        "name": "Full TALMAS",
        "lambda_max": 4.0,
        "use_timestep_gate": True,
        "use_layer_gate": True,
        "mu": 0.1,   # sweep [0, 0.2, 0.5, 1.0] — 0.1 default
        "description": "Full joint gating with partial mask->mask suppression"
    },
]

MU_SWEEP = [0.0, 0.2, 0.5, 1.0]   # for config 5 only
LAMBDA_MAX_DEFAULT = 4.0            # tune if needed; start here

print("Ablation configs:")
for c in ABLATION_CONFIGS:
    print(f"  [{c['id']}] {c['name']}: lambda_max={c['lambda_max']}, mu={c['mu']}")

In [ ]:
# Cell 6 — Benchmark Loaders

def load_gsm8k(n_samples=200):
    """Load GSM8K test split."""
    ds = load_dataset("gsm8k", "main", split="test")
    return [{"question": ex["question"], "answer": ex["answer"]}
            for ex in ds.select(range(n_samples))]

def extract_gsm8k_answer(text: str) -> str:
    """Extract the final numeric answer after ####."""
    match = re.search(r"####\s*([\d,\.\-]+)", text)
    return match.group(1).replace(",", "").strip() if match else ""

def load_humaneval(n_samples=50):
    """Load HumanEval problems."""
    from human_eval.data import read_problems
    problems = read_problems()
    keys = list(problems.keys())[:n_samples]
    return [{"task_id": k, **problems[k]} for k in keys]

print("Benchmark loaders defined.")

In [ ]:
# Cell 7 — Evaluation Loop

def make_gsm8k_prompt(question, tokenizer):
    content = (f"{question}\n\nSolve step by step. "
               "At the end write your final answer as:\n#### <number>")
    messages = [{"role": "user", "content": content}]
    return tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False)


def run_ablation(config, benchmark="gsm8k", n_samples=100,
                 num_steps=64, gen_len=256, debug_first=0):
    print(f"\n{'='*60}")
    print(f"Config {config['id']}: {config['name']}")
    print(f"  {config['description']}")
    print(f"  steps={num_steps}, gen_len={gen_len}")
    print(f"{'='*60}")

    hook_manager = TALMASHookManager(
        model=model, lambda_max=config["lambda_max"], mu=config["mu"],
        use_timestep_gate=config["use_timestep_gate"],
        use_layer_gate=config["use_layer_gate"],
    ) if config["lambda_max"] > 0 else None

    correct = 0
    total = 0
    results = []

    try:
        if benchmark == "gsm8k":
            samples = load_gsm8k(n_samples)
            for i, ex in enumerate(tqdm(samples, desc=config["name"])):
                prompt = make_gsm8k_prompt(ex["question"], tokenizer)
                prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
                gen_ids = llada_generate(
                    model, tokenizer, prompt_ids,
                    gen_len=gen_len, num_steps=num_steps,
                    hook_manager=hook_manager)
                gen_text   = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
                pred       = extract_gsm8k_answer(gen_text)
                gold       = extract_gsm8k_answer(ex["answer"])
                is_correct = (pred == gold)
                correct   += int(is_correct)
                total     += 1
                results.append({"pred": pred, "gold": gold, "correct": is_correct})
                if i < debug_first:
                    print(f"\n--- Sample {i} ---")
                    print(f"GOLD: {gold}  PRED: {pred}")
                    print(gen_text[:300])

        elif benchmark == "humaneval":
            from human_eval.data import read_problems
            from human_eval.execution import check_correctness
            passed = 0
            for i, prob in enumerate(tqdm(load_humaneval(n_samples), desc=config["name"])):
                prompt_ids = tokenizer(prob["prompt"], return_tensors="pt").input_ids.to(model.device)
                gen_ids = llada_generate(
                    model, tokenizer, prompt_ids,
                    gen_len=512, num_steps=num_steps,
                    hook_manager=hook_manager)
                completion = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
                result = check_correctness(prob, completion, timeout=10.0)
                passed += int(result["passed"])
                total  += 1
                results.append({"task_id": prob["task_id"], "passed": result["passed"]})
            correct = passed

    finally:
        # Always remove hooks, even if an exception occurred mid-run
        if hook_manager:
            hook_manager.remove()

    accuracy = correct / total if total > 0 else 0.0
    print(f"  → Accuracy: {accuracy:.3f} ({correct}/{total})")
    return {
        "config_id": config["id"], "config_name": config["name"],
        "lambda_max": config["lambda_max"], "mu": config["mu"],
        "use_timestep_gate": config["use_timestep_gate"],
        "use_layer_gate": config["use_layer_gate"],
        "benchmark": benchmark, "n_samples": total,
        "accuracy": accuracy, "pass_at_1": correct / total if total > 0 else 0.0,
        "raw_results": results,
    }

print("run_ablation defined.")

In [ ]:
# Cell 8 — Run All Ablations

# Force-remove any stale patches from previous runs or Cell 3b tests
talmas_cleanup(model)

N_SAMPLES = 100
NUM_STEPS = 64
GEN_LEN   = 256

all_results = []

for config in ABLATION_CONFIGS:
    if config["id"] == 5:
        for mu in MU_SWEEP:
            cfg = dict(config)
            cfg["mu"]   = mu
            cfg["name"] = f"Full TALMAS (mu={mu})"
            result = run_ablation(cfg, benchmark="gsm8k",
                                  n_samples=N_SAMPLES, num_steps=NUM_STEPS,
                                  gen_len=GEN_LEN)
            all_results.append(result)
    else:
        result = run_ablation(config, benchmark="gsm8k",
                              n_samples=N_SAMPLES, num_steps=NUM_STEPS,
                              gen_len=GEN_LEN)
        all_results.append(result)

df = pd.DataFrame([{k: v for k, v in r.items() if k != "raw_results"}
                   for r in all_results])
df.to_csv("talmas_results.csv", index=False)
print("\nResults saved to talmas_results.csv")
print(df.to_string())

In [ ]:
# Cell 9 — Visualization

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Ablation comparison (configs 1-5, mu=0.1 for config 5)
main_results = df[~df["config_name"].str.contains("mu=")]
ax = axes[0]
bars = ax.bar(range(len(main_results)), main_results["accuracy"],
              color=["#888", "#4C9BE8", "#E87B4C", "#50C878", "#9B59B6"])
ax.set_xticks(range(len(main_results)))
ax.set_xticklabels(main_results["config_name"], rotation=20, ha="right")
ax.set_ylabel("GSM8K Accuracy")
ax.set_title("TALMAS Ablation Study — GSM8K")
ax.set_ylim(0, 1)
for bar, val in zip(bars, main_results["accuracy"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9)

# Plot 2: mu sweep for Full TALMAS
mu_results = df[df["config_name"].str.contains("mu=")]
ax2 = axes[1]
ax2.plot(MU_SWEEP[:len(mu_results)], mu_results["accuracy"].values,
         "o-", color="#9B59B6", linewidth=2, markersize=8)
ax2.set_xlabel("mu (mask->mask suppression scale)")
ax2.set_ylabel("GSM8K Accuracy")
ax2.set_title("Full TALMAS: mu Sweep")
ax2.set_xticks(MU_SWEEP)

plt.tight_layout()
plt.savefig("talmas_ablation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to talmas_ablation.png")

In [ ]:
# Cell 10 — Gate Visualization (diagnostic)
# Run this BEFORE the full ablation to verify gates behave correctly:
#   - Left edge (fully masked)   → lambda ≈ 0  for all layers
#   - Right edge (fully revealed) → lambda ≈ lambda_max for deep layers
#   - Shallow layers               → lambda ≈ 0  for all timesteps

num_layers_vis = 32   # LLaDA-8B
num_steps_vis  = 20
lambda_max_vis = LAMBDA_MAX_DEFAULT

grid = np.zeros((num_layers_vis, num_steps_vis))
for layer in range(num_layers_vis):
    for step_idx, step in enumerate(range(num_steps_vis - 1, -1, -1)):
        r_t = step / max(num_steps_vis - 1, 1)
        grid[layer, step_idx] = compute_lambda(lambda_max_vis, r_t,
                                               layer, num_layers_vis)

plt.figure(figsize=(10, 5))
plt.imshow(grid, aspect="auto", origin="lower",
           extent=[0, 1, 0, num_layers_vis], cmap="viridis")
plt.colorbar(label="lambda(l, t)")
plt.xlabel("Denoising step (left=fully masked, right=fully revealed)")
plt.ylabel("Layer index")
plt.title(f"TALMAS lambda(l, t) Gate — lambda_max={lambda_max_vis}")
plt.savefig("talmas_lambda_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to talmas_lambda_grid.png")

# Sanity check values
print(f"\nCorner values (expect ~0 at left/shallow, ~lambda_max at right/deep):")
print(f"  lambda(layer=0,  fully_masked)   = {grid[0,  0]:.4f}")
print(f"  lambda(layer=0,  fully_revealed) = {grid[0,  -1]:.4f}")
print(f"  lambda(layer=31, fully_masked)   = {grid[31, 0]:.4f}")
print(f"  lambda(layer=31, fully_revealed) = {grid[31, -1]:.4f}")

In [ ]:
# Cell 11 — Confirmation Run: Baseline vs Full TALMAS (μ=0.5) on GSM8K n=500
# Establishes statistical significance of the +7pp gain observed in the ablation.
# Expected runtime on A100: ~45-60 min per config (~1.5 h total).

talmas_cleanup(model)

_baseline_cfg = ABLATION_CONFIGS[0]  # Config 1: lambda_max=0

_talmas_cfg = dict(ABLATION_CONFIGS[4])  # Config 5: Full TALMAS
_talmas_cfg["mu"]   = 0.5
_talmas_cfg["name"] = "Full TALMAS (mu=0.5)"

CONFIRM_N      = 500
CONFIRM_STEPS  = 64
CONFIRM_GENLEN = 256

confirm_results = []

for cfg in [_baseline_cfg, _talmas_cfg]:
    r = run_ablation(cfg, benchmark="gsm8k",
                     n_samples=CONFIRM_N,
                     num_steps=CONFIRM_STEPS,
                     gen_len=CONFIRM_GENLEN)
    confirm_results.append(r)

# ── Statistical test (two-proportion z-test) ─────────────────────────────────
import math

_b = confirm_results[0]
_t = confirm_results[1]
n_b, k_b = _b["n_samples"], int(round(_b["accuracy"] * _b["n_samples"]))
n_t, k_t = _t["n_samples"], int(round(_t["accuracy"] * _t["n_samples"]))
p_b, p_t = k_b / n_b, k_t / n_t
p_pool   = (k_b + k_t) / (n_b + n_t)
se       = math.sqrt(p_pool * (1 - p_pool) * (1/n_b + 1/n_t))
z        = (p_t - p_b) / se if se > 0 else 0.0
# Two-tailed p-value approximation
p_val = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))

print(f"\n{'='*50}")
print(f"Baseline      : {p_b:.3f}  ({k_b}/{n_b})")
print(f"Full TALMAS   : {p_t:.3f}  ({k_t}/{n_t})")
print(f"Delta         : {p_t - p_b:+.3f}")
print(f"Z-score       : {z:.2f}")
print(f"p-value (2-tail): {p_val:.4f}  {'*** p<0.01' if p_val<0.01 else '** p<0.05' if p_val<0.05 else '* p<0.10' if p_val<0.10 else '(not significant)'}")
print(f"{'='*50}")

df_confirm = pd.DataFrame([{k: v for k, v in r.items() if k != "raw_results"}
                            for r in confirm_results])
df_confirm["z_score"] = [0.0, z]
df_confirm["p_value"] = [1.0, p_val]
df_confirm.to_csv("talmas_gsm8k_confirmation.csv", index=False)
print("\nSaved to talmas_gsm8k_confirmation.csv")
print(df_confirm[["config_name", "n_samples", "accuracy", "z_score", "p_value"]].to_string())

In [ ]:
# Cell 12 — HumanEval: Baseline vs Full TALMAS (μ=0.5)
# Tests whether the GSM8K gain generalises to code generation (pass@1).
# HumanEval has 164 problems; we use all of them.
# Expected runtime on A100: ~25-35 min per config (~1 h total).

talmas_cleanup(model)

# human-eval ships its own execution sandbox; make sure it's installed (Cell 0).
try:
    from human_eval.data import read_problems
    from human_eval.execution import check_correctness
    _he_available = True
except ImportError:
    print("human-eval not installed — run Cell 0 first.")
    _he_available = False

if _he_available:
    _he_baseline_cfg = ABLATION_CONFIGS[0]
    _he_talmas_cfg   = dict(ABLATION_CONFIGS[4])
    _he_talmas_cfg["mu"]   = 0.5
    _he_talmas_cfg["name"] = "Full TALMAS (mu=0.5)"

    HE_N_SAMPLES = 164   # full HumanEval benchmark
    HE_STEPS     = 64
    HE_GENLEN    = 512   # code completions need more room

    he_results = []
    for cfg in [_he_baseline_cfg, _he_talmas_cfg]:
        r = run_ablation(cfg, benchmark="humaneval",
                         n_samples=HE_N_SAMPLES,
                         num_steps=HE_STEPS,
                         gen_len=HE_GENLEN)
        he_results.append(r)

    # ── Summary ───────────────────────────────────────────────────────────────
    import math
    _hb = he_results[0]
    _ht = he_results[1]
    n_hb, k_hb = _hb["n_samples"], int(round(_hb["pass_at_1"] * _hb["n_samples"]))
    n_ht, k_ht = _ht["n_samples"], int(round(_ht["pass_at_1"] * _ht["n_samples"]))
    p_hb, p_ht = k_hb / n_hb, k_ht / n_ht
    p_pool_h   = (k_hb + k_ht) / (n_hb + n_ht)
    se_h       = math.sqrt(p_pool_h * (1 - p_pool_h) * (1/n_hb + 1/n_ht))
    z_h        = (p_ht - p_hb) / se_h if se_h > 0 else 0.0
    p_val_h    = 2 * (1 - 0.5 * (1 + math.erf(abs(z_h) / math.sqrt(2))))

    print(f"\n{'='*50}")
    print(f"HumanEval pass@1")
    print(f"Baseline    : {p_hb:.3f}  ({k_hb}/{n_hb})")
    print(f"Full TALMAS : {p_ht:.3f}  ({k_ht}/{n_ht})")
    print(f"Delta       : {p_ht - p_hb:+.3f}")
    print(f"Z-score     : {z_h:.2f}")
    print(f"p-value     : {p_val_h:.4f}  {'*** p<0.01' if p_val_h<0.01 else '** p<0.05' if p_val_h<0.05 else '* p<0.10' if p_val_h<0.10 else '(not significant)'}")
    print(f"{'='*50}")

    df_he = pd.DataFrame([{k: v for k, v in r.items() if k != "raw_results"}
                           for r in he_results])
    df_he["z_score"] = [0.0, z_h]
    df_he["p_value"] = [1.0, p_val_h]
    df_he.to_csv("talmas_humaneval.csv", index=False)
    print("\nSaved to talmas_humaneval.csv")
    print(df_he[["config_name", "n_samples", "pass_at_1", "z_score", "p_value"]].to_string())

In [ ]:
# Cell 13 — Final Summary Plot: GSM8K (n=500) + HumanEval side by side

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
_colors = {"Baseline (LLaDA)": "#888888", "Full TALMAS (mu=0.5)": "#9B59B6"}

for ax, (df_plot, metric, title, ylabel) in zip(axes, [
    (df_confirm, "accuracy",  "GSM8K (n=500)",        "Accuracy"),
    (df_he,      "pass_at_1", "HumanEval (n=164)",    "pass@1"),
]):
    names = df_plot["config_name"].tolist()
    vals  = df_plot[metric].tolist()
    p_vals = df_plot["p_value"].tolist()
    colors = [_colors.get(n, "#4C9BE8") for n in names]
    bars = ax.bar(names, vals, color=colors, width=0.4)
    ax.set_ylim(0, max(vals) * 1.25)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=10, ha="right")
    for bar, val, pv in zip(bars, vals, p_vals):
        sig = "***" if pv < 0.01 else "**" if pv < 0.05 else "*" if pv < 0.10 else ""
        label = f"{val:.3f}{sig}"
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                label, ha="center", va="bottom", fontsize=10)

plt.suptitle("TALMAS vs Baseline — Confirmation + Generalisation", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("talmas_final_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to talmas_final_summary.png")

# ── Combined summary table ────────────────────────────────────────────────────
print("\n=== FINAL RESULTS SUMMARY ===")
for label, df_s, metric in [("GSM8K n=500", df_confirm, "accuracy"),
                              ("HumanEval",  df_he,      "pass_at_1")]:
    base  = df_s[df_s["config_name"] == "Baseline (LLaDA)"][metric].values[0]
    best  = df_s[df_s["config_name"] == "Full TALMAS (mu=0.5)"][metric].values[0]
    pv    = df_s[df_s["config_name"] == "Full TALMAS (mu=0.5)"]["p_value"].values[0]
    delta = best - base
    sig   = "***" if pv < 0.01 else "**" if pv < 0.05 else "*" if pv < 0.10 else "n.s."
    print(f"  {label:20s}: baseline={base:.3f}  TALMAS={best:.3f}  Δ={delta:+.3f}  {sig}")